In [20]:
# Performs analysis of average variograms for 4 seasons 
# within each cluster to derive optimal parameters for kriging
#  Fix 1| 26 April | using ERA5L data as input for kriging instead of insitu data

# Workflow steps:
# 1. Create single seasonal raster from all daily images
# 2. Dissolve pixels to points and transform projections to UTM 45
# * this step onwards require cluster + season subsetting 
# 3. Detect linear trend and create residuals
# 4. Apply nscore transform (check how it works out before and after)
# 5. Isotropy check (check directional variogram; only flag anisotropy and do not actually 
# extend it to the workflow since we are assuming isotropy within cluster)
# 6. Create isotropy assumed experimental variogram with WLS and tweak parameters 
# 7. Perfrom LOOCV to validate parameter choices and select the best visually + metrically
# 8. Store best parameters in a csv

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from pyproj import Proj
import geostatspy.geostats as geostats
import geostatspy.GSLIB as GSLIB

import os
import glob
import rasterio
import rioxarray as rxr
import xarray as xr
from rasterio.enums import Resampling
import dask
import sys

from sklearn.linear_model import LinearRegression

#Apply variables
season = 'monsoon'
cluster_id = 1         #pick among 0 to 3
r2_threshold = 0.05    #trend removal 

#define seasons
seasons = {
        'winter': [12, 1, 2],
        'pre_monsoon': [3, 4, 5],
        'monsoon': [6, 7, 8, 9],
        'post_monsoon': [10, 11]
    }

temp_file_path = r"..\Data\Processed\Variogram_analysis\Temp"
os.makedirs(temp_file_path, exist_ok = True)   

In [ ]:
#1. Create single seasonal raster from all daily images 
#uncomment and run if required

# era5L_files_path = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\ERA5L_prec"
# era5L_files = sorted(glob.glob(os.path.join(era5L_files_path, "*.tif")))
# era5L_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in era5L_files]
# era5L_stack = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in era5L_files], 
#                 dim=pd.Index(era5L_dates, name='time'), join = 'exact')
# era5L_stack = era5L_stack.squeeze('band', drop = True)


# for season, months in seasons.items():
#     mask = era5L_stack.time.dt.month.isin(months)
#     stack = era5L_stack.isel(time=mask)

#     if season == 'winter':
#         time_index = pd.DatetimeIndex(stack.time.values)
#         season_year = np.where(
#             time_index.month == 12,
#             time_index.year + 1,
#             time_index.year
#         )
#         stack = stack.assign_coords(season_year=('time', season_year))
#     else:
#         season_year = pd.DatetimeIndex(stack.time.values).year
#         stack = stack.assign_coords(season_year=('time', season_year))

#     annual_sums = stack.groupby('season_year').sum(dim='time', skipna=True)
#     climatology  = annual_sums.mean(dim='season_year', skipna=True)
#     climatology.rio.to_raster(os.path.join(temp_file_path, f"{season}_clim.tif"))

# del era5L_stack, stack, annual_sums, climatology


In [49]:
# We'll implement seasonal and cluster filter onwards this during execution
# 2. Dissolve pixels to points and transform projections to UTM 45
climatology_rxr = rxr.open_rasterio(os.path.join(temp_file_path, f"{season}_clim.tif"))
climatology_rxr = climatology_rxr.squeeze('band', drop = True)
cluster_rxr = rxr.open_rasterio(r"..\Data\Processed\Spatial_clustering\Precipitation_clusters.tif")
cluster_coarse = cluster_rxr.rio.reproject_match(
        climatology_rxr,
        resampling=Resampling.mode  
    )
cluster_mask = (cluster_coarse == cluster_id)
era5l_cluster = climatology_rxr.where(cluster_mask)
climatology_df = era5l_cluster.to_dataframe(name = "prec").reset_index()
climatology_df = climatology_df.drop(columns= ['spatial_ref','band']).dropna().reset_index()

myProj = Proj("+proj=utm +zone=45 +ellps=WGS84 +datum=WGS84 +units=m +no_defs")
x, y = myProj(climatology_df['x'].values, climatology_df['y'].values)
climatology_df['x_km'] = x / 1000.0
climatology_df['y_km'] = y / 1000.0
climatology_df = climatology_df[['x_km','y_km','prec']]

In [62]:
# 3. Detect linear trend and create residuals

X_coord = climatology_df[['x_km', 'y_km']].values
y_prec = climatology_df['prec'].values
lm = LinearRegression().fit(X_coord, y_prec)
predictions = lm.predict(X_coord)
r2 = lm.score(X_coord, y_prec)
print(f"Trend Surface Fit (R²): {r2:.4f}")
print(f"Linear regression equation: y = B1 (lat) + B2 (lon) + intercept ==", 
            f" y = {lm.coef_[0]:.4f}*(long) + {lm.coef_[1]:.4f}*(lat) + {lm.intercept_:.4f}")

Trend Surface Fit (R²): 0.0175
Linear regression equation: y = B1 (lat) + B2 (lon) + intercept ==  y = -0.2864*(long) + -1.1321*(lat) + 5815.0895


In [ ]:
# 4. Apply nscore transform (check how it works out before and after)
if r2 >= r2_threshold:
    residuals = predictions - y_prec    
else:
    residuals = y_prec
    

[ 6.86150145e+02  6.78038151e+02  3.19155235e+02  2.68608687e+02
  2.39271943e+02  3.88441387e+02  4.55075201e+01 -7.36277087e+01
 -1.71960222e+02 -2.02979807e+02 -3.45761945e+01  2.56212641e+02
 -1.19915090e+02 -2.52943749e+02 -3.50004483e+02 -3.65365603e+02
 -1.88252836e+02  1.12973410e+02  1.22489136e+02  9.42550408e+01
  1.04582742e+02  1.46434043e+02  1.91182459e+02  8.45216386e+01
  6.29286268e+01 -2.23015948e+02 -4.96454330e+02 -7.04721499e+02
 -4.25781987e+02 -8.05430471e+01  2.54504610e+02 -2.76685572e+02
 -6.75918324e+02 -1.08923784e+03 -1.34729293e+03 -1.06125479e+03
 -6.06112071e+02 -1.88196481e+02  7.36051513e+01 -1.05423322e+02
 -4.84352498e+02 -8.78740804e+02 -1.08143999e+03 -8.69827795e+02
 -4.89847870e+02 -1.50158608e+02  6.40057987e+01  2.07159948e+02
  2.43812714e+02  1.56760435e+02 -1.09781422e+02 -2.55721544e+02
 -1.84502801e+02  1.46429670e-01  1.61581621e+02  1.88474626e+02
  1.14322456e+02  6.07177849e+01  1.74397986e+02  8.84613862e+01
 -7.77757042e+00 -1.03718